In [1]:
import torch
print(torch.cuda.is_available())  # True이면 GPU 사용 가능
print(torch.cuda.get_device_name(0))  # GPU 이름 확인


True
Tesla T4


In [3]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.8 MB/s eta 0:00:00


In [4]:

import os
import random
import glob
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from catboost import CatBoostRegressor
import torch
import torch.nn as nn
from tqdm import tqdm

# -------------------- 시드 설정 --------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# -------------------- 하이퍼파라미터 --------------------
LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 16, 50
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -------------------- 매장/메뉴 인덱스 --------------------
store_names = ['느티나무','담하','라그로타','미라시아','연회장','카페테리아','포레스트릿','화담숲주막','화담숲카페']
store_to_index = {name:i for i, name in enumerate(store_names)}

def extract_store_and_menu(full_name, store_names, store_to_index):
    for store in store_names:
        if full_name.startswith(store):
            menu = full_name[len(store):].strip().replace('_','')
            return store_to_index[store], menu
    return -1, full_name

# -------------------- LSTM 모델 --------------------
weekday_vocab_size = 7
month_vocab_size = 12
store_vocab_size = len(store_names)
embed_dim = 4
INPUT_DIM = 1 + embed_dim*3

class MultiOutputLSTM(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=256, num_layers=2, output_dim=PREDICT):
        super().__init__()
        self.weekday_embedding = nn.Embedding(weekday_vocab_size, embed_dim)
        self.month_embedding = nn.Embedding(month_vocab_size, embed_dim)
        self.store_embedding = nn.Embedding(store_vocab_size, embed_dim)
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, sales_seq, weekday_idx_seq, month_idx_seq, store_idx_seq):
        weekday_emb = self.weekday_embedding(weekday_idx_seq)
        month_emb = self.month_embedding(month_idx_seq)
        store_emb = self.store_embedding(store_idx_seq)
        x = torch.cat([sales_seq, weekday_emb, month_emb, store_emb], dim=2)
        out,_ = self.lstm(x)
        out = self.dropout(out)
        out = out.mean(dim=1)
        return self.fc(out)

# -------------------- 손실함수 --------------------
class WeightedHuberLoss(nn.Module):
    def __init__(self, weight=3.0, delta=1.0):
        super().__init__()
        self.huber = nn.SmoothL1Loss(reduction='none', beta=delta)
        self.weight = weight
    def forward(self, pred, true):
        loss = self.huber(pred,true)
        weight_mask = (true>0).float()*self.weight + 1.0
        return (loss*weight_mask).mean()

class SMAPELoss(nn.Module):
    def forward(self, pred,true):
        epsilon = 1e-6
        numerator = torch.abs(true-pred)
        denominator = (torch.abs(true)+torch.abs(pred))/2 + epsilon
        return (numerator/denominator).mean()

# -------------------- LSTM 학습 --------------------
def train_lstm(train_embedded):
    trained_models = {}
    for store_menu, group in tqdm(train_embedded.groupby('영업장명_메뉴명'), desc='Training LSTM'):
        store_train = group.sort_values('영업일자').copy()
        if len(store_train)<LOOKBACK+PREDICT: continue
        train_data = store_train.iloc[:-PREDICT].copy()
        scaler = MinMaxScaler()
        train_data['매출수량'] = scaler.fit_transform(train_data[['매출수량']])
        X_sales,X_weekday,X_month,X_store,y_train=[],[],[],[],[]
        sales_seq = train_data['매출수량'].values
        weekday_seq = train_data['요일'].values
        month_seq = train_data['월_adj'].values
        store_idx_seq = train_data['store_idx'].values
        for i in range(len(train_data)-LOOKBACK-PREDICT+1):
            X_sales.append(sales_seq[i:i+LOOKBACK])
            X_weekday.append(weekday_seq[i:i+LOOKBACK])
            X_month.append(month_seq[i:i+LOOKBACK])
            X_store.append(store_idx_seq[i:i+LOOKBACK])
            y_train.append(sales_seq[i+LOOKBACK:i+LOOKBACK+PREDICT])
        X_sales = torch.tensor(np.array(X_sales)).unsqueeze(-1).float().to(DEVICE)
        X_weekday = torch.tensor(np.array(X_weekday)).long().to(DEVICE)
        X_month = torch.tensor(np.array(X_month)).long().to(DEVICE)
        X_store = torch.tensor(np.array(X_store)).long().to(DEVICE)
        y_train = torch.tensor(np.array(y_train)).float().to(DEVICE)
        model = MultiOutputLSTM(hidden_dim=256).to(DEVICE)
        embedding_params = list(model.weekday_embedding.parameters()) + \
                           list(model.month_embedding.parameters()) + \
                           list(model.store_embedding.parameters())
        embedding_param_ids = set(map(id, embedding_params))
        other_params = [p for p in model.parameters() if id(p) not in embedding_param_ids]
        optimizer = torch.optim.Adam([{'params':embedding_params,'lr':0.0001},
                                      {'params':other_params,'lr':0.001}])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.7, patience=3)
        criterion_huber = WeightedHuberLoss()
        criterion_smape = SMAPELoss()
        model.train()
        for epoch in range(EPOCHS):
            idx = torch.randperm(len(X_sales))
            epoch_loss = 0
            for i in range(0,len(X_sales),BATCH_SIZE):
                batch_idx = idx[i:i+BATCH_SIZE]
                optimizer.zero_grad()
                output = model(X_sales[batch_idx], X_weekday[batch_idx], X_month[batch_idx], X_store[batch_idx])
                loss = 0.5*criterion_huber(output,y_train[batch_idx]) + 0.5*criterion_smape(output,y_train[batch_idx])
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
                epoch_loss += loss.item()
            scheduler.step(epoch_loss/len(X_sales))
        trained_models[store_menu] = {'model':model.eval(),'scaler':scaler}
    return trained_models

# -------------------- CatBoost 학습 (스케일 적용) --------------------
def train_catboost(train_embedded):
    cat_models = {}
    for store_menu, group in tqdm(train_embedded.groupby('영업장명_메뉴명'), desc='Training CatBoost'):
        store_train = group.sort_values('영업일자').copy()
        if len(store_train)<LOOKBACK+PREDICT: continue
        train_data = store_train.iloc[:-PREDICT].copy()
        scaler = MinMaxScaler()
        train_data['매출수량'] = scaler.fit_transform(train_data[['매출수량']])

        df = train_data.copy()
        for lag in range(1,LOOKBACK+1):
            df[f'lag_{lag}'] = df['매출수량'].shift(lag)
        df.dropna(inplace=True)

        X = df[[f'lag_{lag}' for lag in range(1,LOOKBACK+1)] + ['요일','월_adj','store_idx']]
        y = df['매출수량']
        cat_features = ['요일','월_adj','store_idx']

        model = CatBoostRegressor(iterations=200, learning_rate=0.1, depth=6, loss_function='MAPE', verbose=0)
        model.fit(X, y, cat_features=cat_features)

        cat_models[store_menu] = {'model': model, 'scaler': scaler}

    return cat_models

# -------------------- 앙상블 예측 --------------------
def predict_ensemble(trained_lstm, trained_cat, test_files):
    results = []

    for path in tqdm(test_files, desc='Predicting Ensemble'):
        test_df_raw = pd.read_csv(path)
        test_df_raw['영업일자'] = pd.to_datetime(test_df_raw['영업일자'])

        for store_menu in trained_lstm.keys():
            lstm_model = trained_lstm[store_menu]['model']
            lstm_scaler = trained_lstm[store_menu]['scaler']

            cat_info = trained_cat.get(store_menu, None)
            if cat_info is not None:
                cat_model = cat_info['model']
                cat_scaler = cat_info['scaler']
            else:
                cat_model = None
                cat_scaler = lstm_scaler

            store_test = test_df_raw[test_df_raw['영업장명_메뉴명'] == store_menu].sort_values('영업일자').copy()
            if len(store_test) < LOOKBACK:
                continue

            store_test['요일'] = store_test['영업일자'].dt.weekday
            store_test['월_adj'] = store_test['영업일자'].dt.month - 1
            store_test['store_idx'], _ = zip(*store_test['영업장명_메뉴명'].apply(lambda x: extract_store_and_menu(x, store_names, store_to_index)))

            # --- LSTM 예측 ---
            last_sequence = store_test['매출수량'].values[-LOOKBACK:]
            last_seq_df = pd.DataFrame(last_sequence.reshape(-1, 1), columns=lstm_scaler.feature_names_in_)
            last_sequence_scaled = lstm_scaler.transform(last_seq_df).flatten()

            x_sales = torch.tensor(last_sequence_scaled).unsqueeze(0).unsqueeze(-1).float().to(DEVICE)
            x_weekday = torch.tensor(store_test['요일'].values[-LOOKBACK:]).unsqueeze(0).long().to(DEVICE)
            x_month = torch.tensor(store_test['월_adj'].values[-LOOKBACK:]).unsqueeze(0).long().to(DEVICE)
            x_store = torch.tensor(store_test['store_idx'].values[-LOOKBACK:]).unsqueeze(0).long().to(DEVICE)

            with torch.no_grad():
                lstm_pred_scaled = lstm_model(x_sales, x_weekday, x_month, x_store).squeeze().cpu().numpy()

            restored_lstm = [
                max(lstm_scaler.inverse_transform(pd.DataFrame([[val]], columns=lstm_scaler.feature_names_in_))[0,0],0)
                for val in lstm_pred_scaled
            ]

            # --- CatBoost 예측 ---
            restored_cat = []
            if cat_model is not None:
                last_values = last_sequence_scaled.tolist()
                weekday_seq = store_test['요일'].values[-LOOKBACK:].tolist()
                month_seq = store_test['월_adj'].values[-LOOKBACK:].tolist()
                store_idx_seq = store_test['store_idx'].values[-LOOKBACK:].tolist()

                for i in range(PREDICT):
                    features = last_values[-LOOKBACK:] + [weekday_seq[-1], month_seq[-1], store_idx_seq[-1]]
                    X_pred = pd.DataFrame([features], columns=[f'lag_{l+1}' for l in range(LOOKBACK)] + ['요일','월_adj','store_idx'])
                    pred_scaled = cat_model.predict(X_pred)[0]
                    restored_val = max(cat_scaler.inverse_transform(pd.DataFrame([[pred_scaled]], columns=cat_scaler.feature_names_in_))[0,0],0)
                    restored_cat.append(restored_val)
                    last_values.append(pred_scaled)
            else:
                restored_cat = restored_lstm

            # --- 앙상블 ---
            ensemble_pred = [0.6*l + 0.4*c for l,c in zip(restored_lstm, restored_cat)]

            # --- 날짜 생성 ---
            end_date = store_test['영업일자'].max()
            pred_start_date = end_date + pd.Timedelta(days=1)
            pred_dates = pd.date_range(start=pred_start_date, periods=PREDICT).strftime('%Y-%m-%d').tolist()

            for date, val in zip(pred_dates, ensemble_pred):
                results.append({'영업일자': date, '영업장명_메뉴명': store_menu, '매출수량': round(val)})

    return pd.DataFrame(results)


In [5]:
# -------------------- 데이터 로드 및 전처리 --------------------
train_raw = pd.read_csv('train.csv')
train_embedded = train_raw.copy()
train_embedded['영업일자'] = pd.to_datetime(train_embedded['영업일자'])
train_embedded['요일'] = train_embedded['영업일자'].dt.weekday
train_embedded['월'] = train_embedded['영업일자'].dt.month
train_embedded['store_idx'], _ = zip(*train_embedded['영업장명_메뉴명'].apply(lambda x: extract_store_and_menu(x, store_names, store_to_index)))
train_embedded['월_adj'] = train_embedded['월']-1
train_embedded['매출수량'] = train_embedded['매출수량'].apply(lambda x:max(x,0))

# -------------------- 모델 학습 --------------------
trained_lstm = train_lstm(train_embedded)
trained_cat = train_catboost(train_embedded)

Training CatBoost: 100%|██████████| 193/193 [01:46<00:00,  1.82it/s]


In [6]:
# -------------------- 테스트셋 예측 --------------------
test_files = sorted(glob.glob('TEST_*.csv'))
if not test_files: print("TEST_*.csv 파일을 찾을 수 없습니다.")

pred_df = predict_ensemble(trained_lstm, trained_cat, test_files)

Predicting Ensemble: 100%|██████████| 10/10 [01:10<00:00,  7.03s/it]


In [7]:
pred_df

,영업일자,영업장명_메뉴명,매출수량
0,2024-07-14,느티나무 셀프BBQ_1인 수저세트,11
1,2024-07-15,느티나무 셀프BBQ_1인 수저세트,11
2,2024-07-16,느티나무 셀프BBQ_1인 수저세트,6
3,2024-07-17,느티나무 셀프BBQ_1인 수저세트,7
4,2024-07-18,느티나무 셀프BBQ_1인 수저세트,10
...,...,...,...
13505,2025-05-27,화담숲카페_현미뻥스크림,30
13506,2025-05-28,화담숲카페_현미뻥스크림,27
13507,2025-05-29,화담숲카페_현미뻥스크림,29
13508,2025-05-30,화담숲카페_현미뻥스크림,28


In [8]:
# -------------------- 제출용 날짜 변환 --------------------
test_start_dates = {
    'TEST_00': '2024-07-14','TEST_01': '2024-08-18','TEST_02': '2024-09-22','TEST_03': '2024-10-27',
    'TEST_04': '2024-12-01','TEST_05': '2025-01-05','TEST_06': '2025-02-09','TEST_07': '2025-03-16',
    'TEST_08': '2025-04-20','TEST_09': '2025-05-25',
}
test_start_dates_dt = sorted([pd.to_datetime(d) for d in test_start_dates.values()])

def convert_date_to_test_label(date):
    date = pd.to_datetime(date)
    for i, start_date in enumerate(test_start_dates_dt):
        if i == len(test_start_dates_dt) - 1 or (date >= start_date and date < test_start_dates_dt[i + 1]):
            day_offset = (date - start_date).days + 1
            test_code = f'TEST_{i:02d}'
            return f'{test_code}+{day_offset}일'
    return None

pred_df['영업일자'] = pred_df['영업일자'].apply(convert_date_to_test_label)

In [9]:
# -------------------- submission 변환 --------------------
sample_submission = pd.read_csv('sample_submission.csv')

def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    pred_dict = dict(zip(
        zip(pred_df['영업일자'], pred_df['영업장명_메뉴명']),
        pred_df['매출수량']
    ))
    final_df = sample_submission.copy()
    for row_idx in final_df.index:
        date = final_df.loc[row_idx, '영업일자']
        for col in final_df.columns[1:]:
            final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
    return final_df

submission = convert_to_submission_format(pred_df, sample_submission)
submission.to_csv('ensemble_submission.csv', index=False, encoding='utf-8-sig')
print("Ensemble submission 생성 완료: ensemble_submission.csv")

Ensemble submission 생성 완료: ensemble_submission.csv
